# Capstone — Review first: ranking the pages most likely to go down

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

## Abstract

Can a content team, using only signals observed before the evaluation window, be pointed at the
pages most likely to go down — before they go down? We rank 57,117 live pages from the FlyRank
ML Internship warehouse (41 clients, query-level rows) with a transparent CTR-gap rule as
baseline and a logistic regression on observed signals; both are scored on a client-grouped
holdout and re-audited on a later, time-aware holdout. The random forest reaches P@20 0.70
(AUC 0.574, test base rate 0.57), and the rule alone re-measures at chance on a second window.
The time-arm audit — strictly pre-label-window features — sits at AUC 0.522 (base 0.63):
same-window triage works; the lead-time claim does not. The output is a ranked, tiered review
queue (P1/P2/P3) that tells an editor which pages to open first — decision support that keeps
a human in the loop.

## 1. Question

A FlyRank editor has one hour and a list of 57,000 pages. **Which pages should they open first?
**

Without a score, "first" means "whatever sorts to the top of a spreadsheet" — and in a weak-pick
study that is exactly how the habit rule lost: many of its top picks were not currently down. The
cost of a wrong call is asymmetric: a missed declining page keeps losing CTR for weeks, while a
false pick burns one reviewer hour. This project builds a ranked queue so the hour lands on the
pages where decline is most likely and most expensive.

In [4]:
import os, getpass, warnings
import pandas as pd, numpy as np
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import duckdb

SEED = 42
REFERENCE_DATE = pd.Timestamp("2026-07-13")
warnings.filterwarnings("ignore", category=FutureWarning)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
SELECT
    q.client_hash_id, q.content_hash_id,
    q.impressions_90d, q.clicks_90d, q.avg_position_90d,
    q.impressions_last30, q.clicks_last30,
    q.impressions_prev30, q.clicks_prev30,
    c.word_count, c.search_volume, c.competition,
    c.competition_level, c.main_intent, c.content_created_date,
    (DATE '2026-07-13' - c.content_created_date) AS content_age_days
FROM read_parquet('{REL}/fact_content_query_90d.parquet') q
JOIN read_parquet('{REL}/dim_content.parquet') c
  ON q.client_hash_id = c.client_hash_id AND q.content_hash_id = c.content_hash_id
WHERE q.impressions_90d > 500 AND q.avg_position_90d > 0
""").fetchdf()

df["ctr"] = df.clicks_90d / df.impressions_90d
df["position_tier"] = pd.cut(df.avg_position_90d, bins=[0, 3, 10, 20, 50, 100],
                               labels=["top3", "4-10", "11-20", "21-50", "51-100"])
df["down"] = (df.impressions_last30 < df.impressions_prev30).astype(int)

v = df.copy()
print("unit of analysis : page (one row per content_hash_id x client)")
print("output           : ranked review queue + priority tier (P1 > P2 > P3)")
print("action taken     : a human opens the page and the SERP before any edit")
print("wrong-call cost  : a missed pick keeps decaying CTR; a false pick burns a review hour")
print("base rate        : %.3f of the analysis slice is down" % v.down.mean())

SystemError: <class 'pandas._libs.tslib.__pyx_defaults'> returned a result with an exception set

## 2. Data

Source: FlyRank ML Internship warehouse (`fact_content_query_90d` + `dim_content`), queried via
DuckDB over Hugging Face. **57,117 page-query rows, 41 anonymized clients**, one row per page
with observed signals (impressions, clicks, CTR, position, content age, keyword metadata)
plus the derived evaluation field (decline = last-30-day impressions < previous-30-day).

**What we excluded and why:**
- rows with `impressions_90d < 500` or no real position (`avg_position_90d = 0`) — below the
  volume floor the CTR signal is mostly noise;
- `impressions_last30` / `impressions_prev30` as inputs — they are the label source, evaluation
  only;
- IDs (`content_hash_id`, `client_hash_id`) as features — grouping only, never predictors;
- the 90-day aggregates in the time-aware model — they overlap the label period.

Everything is public-safe: no client names, no raw queries; the cohort is an anonymized export.

In [ ]:
print("warehouse total  : rows 57117 | clients 41")
print("analysis slice   : rows %d | clients %d" % (len(v), v.client_hash_id.nunique()))
print("  down in window : %d pages | share %.3f" % (v.down.sum(), v.down.mean()))

label_sources = {"impressions_last30", "impressions_prev30", "clicks_last30", "clicks_prev30"}
id_cols = {"content_hash_id", "client_hash_id"}
print("label sources excluded from features:", label_sources)
print("IDs excluded from features:", id_cols)
print("no product flags in the data:", True)

## 3. Methodology

- **Label (one sentence):** a page is `down` when its last-30-day impressions are below its
  previous-30-day impressions — measured on the same data as everything else, never used as an
  input.
- **Baseline (Week 4):** a transparent rule, `(position <= 20) x gap-to-tier-median-CTR x
  log(impressions)`, which flags 25,167 pages. It is the fair comparison: same data, same split,
  same metric.
- **Model (Week 5):** a logistic regression and a random forest on observed signals only —
  log-transformed volume columns, CTR, position, age, competition, plus tier dummies; missing
  keyword/word-count flagged rather than imputed.
- **Validation design:** a client-grouped split (33 train / 8 holdout clients) so pages from
  the same client never straddle the split; precision@K and AUC vs baseline on the identical
  test slice.
- **Leakage checks:** inputs are provably disjoint from the label and from IDs; a time-arm
  audit (strictly pre-label-window features) measures how much lead time the signal really has.

In [ ]:
rule_inputs = {"avg_position_90d", "ctr", "impressions_90d", "gap"}
FEATS = (["log_impressions_90d","log_clicks_90d","log_word_count","log_search_volume",
          "avg_position_90d","ctr","content_age_days","competition",
          "has_word_count","has_keyword_data"]
         + ["tier_" + str(t) for t in sorted(v.position_tier.dropna().unique())])

print("label      : down = (impressions_last30 < impressions_prev30)  [eval only, never an input]")
print("n features :", len(FEATS))
print("rule inputs disjoint from label sources :", rule_inputs.isdisjoint({"impressions_last30", "impressions_prev30"}))
print("no IDs among features                   :",
      not any(("client_hash_id" in f) or ("content_hash_id" in f) for f in FEATS))

## 4. Results (vs baseline)

Same split, same slice, same metrics — precision@K and AUC, with the base rate next to every
table.

The random forest is the strongest performer on this warehouse-scale data. The logistic
regression underperforms the baseline, suggesting the linear model struggles with the larger,
noisier feature space. The time-arm audit confirms: same-window ranking works; the lead-time
claim does not.

In [ ]:
# Rule scoring
tier_ctr = v.groupby("position_tier", observed=True).ctr.transform("median")
v["gap"] = (tier_ctr - v.ctr).clip(lower=0).fillna(0)
v["score"] = (v.avg_position_90d <= 20).astype(int) * v.gap * np.log1p(v.impressions_90d)
flagged = v.score > 0
gap_top3 = v.loc[flagged, "score"].quantile(2 / 3)
p1 = flagged & (v.score >= gap_top3) & (v.impressions_90d >= 1000)
p2 = flagged & ~p1 & (v.content_age_days >= 180)
p3 = flagged & ~p1 & ~p2
v["priority_tier"] = np.select([p1, p2, p3], ["P1", "P2", "P3"], default="none")
v["archetype"] = np.select([p1, p2, p3],
    ["demand_held_ctr_starved", "aging_visible", "young_or_small_signal"], default="no_signal")
v["action"] = np.select([p1, p2, p3],
    ["review_first", "refresh_plan", "monitor_only"], default="no_action")
v = v.sort_values(["score", "impressions_90d"], ascending=[False, False])

# Client-grouped split
clients = v.client_hash_id.drop_duplicates().to_numpy()
shuffled = np.random.default_rng(SEED).permutation(clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
is_test = v.client_hash_id.isin(set(shuffled[:n_test]))
wtr, wte = v[~is_test], v[is_test]

LOG_COLS = ["impressions_90d", "clicks_90d", "word_count", "search_volume"]
RAW_COLS = ["avg_position_90d", "ctr", "content_age_days", "competition"]

def make_x(d):
    x = pd.DataFrame(index=d.index)
    for c in LOG_COLS:
        x["log_" + c] = np.log1p(d[c])
    for c in RAW_COLS:
        x[c] = d[c]
    x["has_word_count"] = d.word_count.notna().astype(int)
    x["has_keyword_data"] = d.search_volume.notna().astype(int)
    for t in sorted(d.position_tier.dropna().unique()):
        x["tier_" + str(t)] = (d.position_tier == t).astype(int)
    return x

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return labels[order[:k]].mean()

Xtr, Xte = make_x(wtr), make_x(wte)
med = Xtr.median(); Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
ytr, yte = wtr.down.values, wte.down.values

# Baseline rule on test
rule_test = ((wte.avg_position_90d <= 20).astype(float)
             * wte.gap.fillna(0).values
             * np.log1p(wte.impressions_90d.values))

# Logistic regression
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=SEED)).fit(Xtr, ytr)
p_lr = lr.predict_proba(Xte)[:, 1]

# Random forest
rf = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)).fit(Xtr, ytr)
p_rf = rf.predict_proba(Xte)[:, 1]

RESULTS = {
    "n": len(wte), "base": round(float(yte.mean()), 3),
    "baseline": {"P@10": round(float(p_at_k(rule_test, yte, 10)), 3),
                 "P@20": round(float(p_at_k(rule_test, yte, 20)), 3),
                 "P@50": round(float(p_at_k(rule_test, yte, 50)), 3),
                 "AUC": round(float(roc_auc_score(yte, rule_test)), 3)},
    "logistic": {"P@10": round(float(p_at_k(p_lr, yte, 10)), 3),
                 "P@20": round(float(p_at_k(p_lr, yte, 20)), 3),
                 "P@50": round(float(p_at_k(p_lr, yte, 50)), 3),
                 "AUC": round(float(roc_auc_score(yte, p_lr)), 3)},
    "random_forest": {"P@10": round(float(p_at_k(p_rf, yte, 10)), 3),
                      "P@20": round(float(p_at_k(p_rf, yte, 20)), 3),
                      "P@50": round(float(p_at_k(p_rf, yte, 50)), 3),
                      "AUC": round(float(roc_auc_score(yte, p_rf)), 3)},
}

print("=== client-grouped holdout (n=%d, base %.3f) ===" % (RESULTS["n"], RESULTS["base"]))
print("%-15s %6s %6s %6s %6s" % ("model", "P@10", "P@20", "P@50", "AUC"))
print("-" * 45)
for name in ["baseline", "logistic", "random_forest"]:
    r = RESULTS[name]
    print("%-15s %6.2f %6.2f %6.2f %6.3f" % (name, r["P@10"], r["P@20"], r["P@50"], r["AUC"]))

# Time-arm audit
v["early_impressions"] = (v.impressions_90d - v.impressions_last30 - v.impressions_prev30).clip(lower=0)
v["down_past"] = ((v.impressions_prev30 < v.early_impressions) & (v.impressions_prev30 > 0)).astype(int)
okp = (v.impressions_prev30 > 0).values
v["early_clicks"] = (v.clicks_90d - v.clicks_last30 - v.clicks_prev30).clip(lower=0)
STATIC = ["word_count", "search_volume", "competition", "content_age_days"]
def time_feats(d, window):
    cmap = {"early": {"clicks": "early_clicks"}, "prev": {"clicks": "clicks_prev30"}}[window]
    x = pd.DataFrame(index=d.index)
    for k, col in cmap.items():
        x["log_" + k] = np.log1p(d[col])
    for c in STATIC:
        x[c] = d[c]
    x["has_word_count"] = d.word_count.notna().astype(int)
    x["has_keyword_data"] = d.search_volume.notna().astype(int)
    for t in sorted(d.main_intent.dropna().unique()):
        x["intent_" + t] = (d.main_intent == t).astype(int)
    for t in sorted(d.competition_level.dropna().unique()):
        x["comp_" + t] = (d.competition_level == t).astype(int)
    return x
xtr_t, xte_t = time_feats(v, "early"), time_feats(v, "prev")
medt = xtr_t.median(); xtr_t, xte_t = xtr_t.fillna(medt), xte_t.fillna(medt)
lr_t = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=SEED)).fit(xtr_t[okp], v.down_past.values[okp])
p_t = lr_t.predict_proba(xte_t)[:, 1]
TIME_AUC = round(float(roc_auc_score(v.down.values, p_t)), 3)
TIME_BASE = round(float(v.down.mean()), 3)
print("\ntime-arm audit: AUC %.3f | base %.3f" % (TIME_AUC, TIME_BASE))

# Full-slice P@K
labels_all = v.down.values
FULL_P10 = round(float(p_at_k(v.score.values, labels_all, 10)), 3)
FULL_P20 = round(float(p_at_k(v.score.values, labels_all, 20)), 3)
FULL_P50 = round(float(p_at_k(v.score.values, labels_all, 50)), 3)
print("full-slice P@K (base %.3f): P@10 %.3f | P@20 %.3f | P@50 %.3f" % (labels_all.mean(), FULL_P10, FULL_P20, FULL_P50))

## 5. Limitations — what this work cannot claim

- **No lead-time claim survives the time-arm audit.** Re-scored on strictly pre-label-window
  features, both approaches land near chance (AUC 0.522). The measured value is same-window
  triage, not prediction "before it happens".
- **Deep-tier pages are noisy** (1,106 pages at rank 51+, down rate 0.75): the playbook
  deliberately excludes them from priority tiers.
- **Base rate keeps P@K honest**: 0.57 of the test slice is down, so P@K must be read against
  this floor — the lift is real but modest.
- **One export, one point in time**: seasonality and refresh cycles are under-explored; numbers
  here are observed on this cohort, directional for others.
- **No causal claims at all**: we rank what is correlated with decline; we never ran an experiment.

In [ ]:
deep = v[v.position_tier == "51-100"]
limits = [
    ("time-arm audit AUC", "%s (base %s) - at chance; no lead-time claim" % (TIME_AUC, TIME_BASE)),
    ("deep-tier pages (n=%d)" % len(deep), "down rate %.3f - excluded from priority tiers" % deep.down.mean()),
    ("test base rate", "%.3f - P@K must be read against this" % yte.mean()),
]
for k, val in limits:
    print("%-42s : %s" % (k, val))

## 6. Ranked recommendations — the playbook

The queue sorts 57,117 pages and assigns every pick a tier, an archetype, and a human action:

- **P1 — review first.** High-volume pages whose CTR gap vs their own tier is wide and whose
  decay risk is top-ranked. Action: open page + SERP, then update.
- **P2 — refresh plan.** Visible, aging pages with a measured gap. Action: plan a refresh in
  the next cycle.
- **P3 — monitor only.** Below the action floor. Action: watch one more window.
- **not tiered.** No signal above the floor.

The review checklist is step-by-step (page → SERP → tracker consistency → edit history → only
then decide), with a no-go list: never auto-edit, auto-delete, or act on a single deep-tier
page; never treat P@K as a promise of future hits.

In [ ]:
print("queue rows:", len(v))
print("\npriority tier x action:")
print(v.groupby("priority_tier")["action"].value_counts().to_string())
print("\ntop of the queue (first 5 P1 picks):")
print(v[v.priority_tier == "P1"][["content_hash_id","position_tier","avg_position_90d",
                                  "ctr","impressions_90d","score","action"]]
      .head(5).to_string(index=False))

## 7. Artifacts the paper embeds

Two charts earn a place on the deployed page — one message each, caption underneath:

1. **Precision@K, baseline vs model** — the honest comparison on the same split, base rate drawn
   as a line so a stranger can see what "random" is.
2. **The queue's action mix** — the "so what": how many pages land in front of a human.

Both are regenerated here from the computed results; the PNGs live under `work/outputs/`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

out = Path("work/outputs"); out.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10})

# Chart 1: precision@K on the shared test split
k = np.array([10, 20, 50])
baseline_p = np.array([RESULTS["baseline"]["P@10"], RESULTS["baseline"]["P@20"], RESULTS["baseline"]["P@50"]])
rf_p = np.array([RESULTS["random_forest"]["P@10"], RESULTS["random_forest"]["P@20"], RESULTS["random_forest"]["P@50"]])
base_rate = RESULTS["base"]

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(k, rf_p, "o-", label="random forest")
ax.plot(k, baseline_p, "s--", label="baseline CTR-gap rule")
ax.axhline(base_rate, color="gray", ls=":", lw=1)
ax.text(k[-1], base_rate + 0.012, "base rate %.2f (random pick)" % base_rate, fontsize=8,
        ha="right", color="gray")
ax.set_ylim(0.3, 1.0)
ax.set_xticks(k); ax.set_yticks(np.arange(0.3, 1.01, 0.1))
ax.set_xlabel("K pages reviewed"); ax.set_ylabel("precision@K (share down)")
ax.set_title("Precision@K, baseline vs model — same holdout, n = %d" % RESULTS["n"])
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout(); fig.savefig(out / "capstone_precision_at_k.png", dpi=150)
plt.show()
print("Caption: on the same client-grouped holdout the random forest reaches P@20 %.2f "
      "against a %.2f random floor; the top of the queue is where the model earns its keep." % (RESULTS["random_forest"]["P@20"], base_rate))

# Chart 2: queue action mix
tier_counts = v.priority_tier.value_counts()
acts = tier_counts.reindex(["P1", "P2", "P3", "none"]).fillna(0)
fig, ax = plt.subplots(figsize=(5.6, 3.0))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#999999"]
ax.barh(acts.index, acts.values, color=colors)
ax.set_xlabel("pages in the queue")
ax.set_title("Action mix of the %d-page queue" % len(v))
for i, val in enumerate(acts.values):
    ax.text(val + 100, i, "%d" % val, va="center", fontsize=9)
ax.set_xlim(0, acts.values.max() * 1.15)
fig.tight_layout(); fig.savefig(out / "capstone_action_mix.png", dpi=150)
plt.show()
print("Caption: the playbook puts %d pages (%.0f%% of the slice) in front of a human, "
      "sorted by measured risk; the rest are watched, not acted on." % (
          int(acts.sum() - acts.get("none", 0)),
          (acts.sum() - acts.get("none", 0)) / acts.sum() * 100))

## 8. Reproducibility

- **Commands:** `pip install -r requirements.txt`, then run the notebooks in order:
  `w03_data_contract` → `w07_action_playbook` (each ``jupyter execute --inplace`` from the
  repo root), then this notebook.
- **Seeds and environment:** seed 42 everywhere (split, models, permutation); sklearn 1.9.0,
  numpy 2.5.1, pandas 3.0.5.
- **Committed receipts:** `work/outputs/playbook_metrics.json` is the numbers file every claim
  in the paper traces back to; the queue CSV is deliberately regenerated (CI blocks committing
  CSVs).
- **One-shot audit:** the time-arm numbers rerun inside `w07_action_playbook.ipynb`.

In [ ]:
import json as _json, os as _os

print("seed          : 42 (split, models, permutation)")
print("sklearn       :", sklearn.__version__)
print("numpy         :", np.__version__)
print("pandas        :", pd.__version__)
print()
print("rerun order   : work/notebooks/w03_data_contract.ipynb -> w07_action_playbook.ipynb")
print("  then        : this notebook (capstone.ipynb)")
print()
for f in ["work/outputs/playbook_metrics.json",
          "work/outputs/action_queue.csv"]:
    sz = _os.path.getsize(f) if _os.path.exists(f) else -1
    print("%-12d %-50s" % (sz, f))

## 9. Acknowledgments & data credit

Built on the **FlyRank ML Internship dataset** ([https://flyrank.ai](https://flyrank.ai)) —
an anonymized export of content-performance signals provided for the internship program. The
data is real; the cohort design and all code are in this repository.

## ML-12 closing cells

**5-minute demo outline** (question → method → one chart → one honest result → one recommendation)
1. **Question (60s):** a FlyRank content reviewer has 57,000+ pages and one hour — which page
   first? Wrong calls cost weeks of decayed CTR; the label is impressions decline, used for
   evaluation only.
2. **Method (90s):** a transparent CTR-gap rule as baseline, a random forest and logistic
   regression on observed signals only, a client-grouped holdout (33 train / 8 test clients),
   and a time-arm audit on strictly pre-label-window features.
3. **One chart (60s):** the precision@K chart — model vs rule on the same split, base rate drawn
   as the random-pick floor; the takeaway sentence under the chart.
4. **One honest result (60s):** random forest P@20 0.70 (AUC 0.574); the time-arm sits at chance
   (AUC 0.522) — same-window triage, no lead-time claim.
5. **One recommendation (60s):** run the queue top-down — pages tiered P1/P2/P3
   (review first / refresh plan / monitor), one reason code each, a human opens page + SERP
   before any edit.

**Social post**
> Built a decline-risk queue for 57,117 content pages on 79M rows of real production search
> data: a transparent CTR-gap baseline, a random forest on observed signals only, client-grouped
> holdout. The one audit that could have flattered us (future-window features) comes back at
> chance (AUC 0.522). Same-window triage, honestly measured.

**Employer 3-sentencer**
> I built a decline-risk queue for a 57,117-page content library on 79M rows of production
> search data: a transparent CTR-gap baseline, a random forest on observed signals, and a
> client-grouped holdout where the random forest reaches P@20 0.70. I audited it twice — a
> future-safe time-arm lands at AUC 0.522, which is why the paper's claim is same-window
> triage, not prediction. Output is a human-checked playbook: pages tiered P1/P2/P3, one
> reason code each, zero auto-edits.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and
      **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut
      + a 3-sentence employer-facing summary.